# Lab 04: Indexing and Retrieval
## Student workbook

**Information Retrieval · Python / Jupyter · Eight programming exercises**

Build a small search engine from saved HTML: extract article text, normalize terms, construct an inverted index, and answer Boolean and ranked queries. The final exercise connects the index to a breadth-first crawler.



Complete the code cells marked **E1–E8** and the written-response spaces. Keep the supplied tests unchanged. Hints identify useful operations without supplying the implementations.

## Learning outcomes and route

By the end of the lab, you should be able to distinguish term, document, and collection frequency; build and validate postings lists; explain AND retrieval; calculate and implement BM25; and separate crawling, parsing, preprocessing, indexing, and ranking.

| Stage | Work | Evidence |
|:--|:--|:--|
| 1–3 | Normalize text, extract articles, and filter URLs | E1–E3 checks |
| 4–5 | Build the index and audit collection statistics | E4–E5 checks and corpus audit |
| 6–7 | Retrieve and rank documents | E6–E7 checks and hand calculation |
| 8 | Connect a breadth-first crawler | E8 checks on a deterministic miniature web |
| Finish | Save, reload, search, and explain limitations | Integration checks and exit questions |

**Prerequisites.** Python functions and classes, dictionaries, sets, lists, loops, `Counter`, and basic logarithms. This lab continues the earlier crawling work; Selenium is not needed to process the saved HTML.

**Offline by default.** The package preserves all 128 supplied HTML snapshots: 100 in `docs_collection` and 28 in `test_collection`. There are repeated canonical URLs and one empty snapshot. These are deliberate audit cases, not 128 distinct articles. The pages are historical source material, not current news. Some original news content concerns violence or abuse; the hand-worked examples use neutral energy topics.

**Important boundary.** Crawling discovers pages. Indexing builds a searchable representation of their text. Ranking operates on the index; it does not download pages again.

## 0. Setup and notebook workflow

Extract the entire ZIP before opening the notebook. Keep `lab_support.py` and the `data` folder beside it. Install the dependencies once in the same Python environment used by the notebook:

```bash
python -m pip install -r requirements.txt
python -m jupyter lab
```

Use a Python 3 kernel. The regex tokenizer and Porter stemmer used here need no downloaded NLTK datasets. After installation, all required exercises run without internet access.

Run cells from top to bottom with **Shift+Enter**. After changing an earlier class, use **Restart Kernel and Run All Cells** so that existing objects do not retain an older implementation. To edit an answer, double-click its Markdown cell, replace the response text, and run the cell again. Inline mathematics uses `$...$`; a displayed equation uses `$$...$$` on separate lines.

The check helper reports **PASS**, **INCOMPLETE**, or **BLOCKED**. An unfinished exercise raises `NotImplementedError`; only that specific exception is treated as incomplete. A failed assertion or another programming error produces a real traceback. A workbook that runs with incomplete checks is **not** a completed submission.

In [1]:
from collections import Counter, deque
from pathlib import Path
from types import SimpleNamespace
from urllib.parse import urljoin, urlsplit, urlunsplit, unquote
from tempfile import TemporaryDirectory
import copy
import html
import math
import json

import nltk
import bs4
from bs4 import BeautifulSoup, SoupStrainer, Comment
from nltk.tokenize import WordPunctTokenizer
from nltk.stem import PorterStemmer
from IPython.display import display, Markdown

from lab_support import (
    ROOT, CHECKS, check, ready, manifest_records, snapshot_bytes,
    validate_index, save_index, load_index, persist_page,
)

CHECKS.clear()
print(f"NLTK {nltk.__version__}; Beautiful Soup {bs4.__version__}")
print("Saved pages:", len(manifest_records()), "+", len(manifest_records("test_collection")))

NLTK 3.9.2; Beautiful Soup 4.14.3
Saved pages: 100 + 28


## 1. A shared text-processing pipeline — E1

A search term is a normalized token, not necessarily a dictionary word. Apply **the same pipeline** to documents and queries:

$$
P(x)=\bigl[\operatorname{stem}(w)\;\bigm|\;
 w\in\operatorname{tokenize}(\operatorname{lower}(x)),\quad
 w.\operatorname{isalpha}(),\quad w\notin S\bigr].
$$

Here $S$ is the fixed stop-word set below. The square brackets represent an ordered sequence: repeated terms are retained because their counts will become term frequencies. Filtering happens **before** stemming. Document length is the number of tokens that remain after this entire pipeline.

Use NLTK's `WordPunctTokenizer`, which splits words and punctuation with a regular expression [3]. This is an explicit change from the starter's downloadable sentence-tokenizer dependency. `isalpha()` accepts alphabetic Unicode tokens; it rejects digits and mixed alphanumeric tokens. The Porter stemmer is an English stemmer, not a multilingual language-understanding model.

### E1 · Implement `Preprocessor` 

Complete `tokenize`, `stem`, `is_apt_word`, and `preprocess`. Preserve the given method signatures and stop-word set. `is_apt_word` should also behave correctly when called directly with an uppercase stop word.

**Hint.** Tokenize the lowercased input, filter each token, and stem the survivors. A `set` is appropriate for stop-word membership, but not for storing the final token sequence.

In [2]:
class Preprocessor:
    def __init__(self):
        self.stop_words = {
            "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
            "has", "he", "in", "is", "it", "its", "of", "on", "that", "the",
            "to", "was", "were", "will", "with",
        }
        self.tokenizer = WordPunctTokenizer()
        self.ps = PorterStemmer()

    def config(self):
        # Persist the exact analysis policy with the index.
        return {"tokenizer": "WordPunctTokenizer", "lowercase": True,
                "alphabetic_only": True, "stemmer": "PorterStemmer",
                "stemmer_mode": self.ps.mode, "nltk_version": nltk.__version__,
                "stop_words": sorted(self.stop_words)}

    def tokenize(self, text: str) -> list[str]:
        # TODO E1a: use the tokenizer already stored on this object.
        raise NotImplementedError("E1a: tokenize the text")

    def stem(self, word: str, stemmer) -> str:
        # TODO E1b: use the supplied stemmer argument.
        raise NotImplementedError("E1b: stem one word")

    def is_apt_word(self, word: str) -> bool:
        # TODO E1c: require letters only and reject stop words.
        raise NotImplementedError("E1c: decide whether a token is kept")

    def preprocess(self, text: str) -> list[str]:
        # TODO E1d: lowercase, tokenize, filter, and stem; retain repeats.
        raise NotImplementedError("E1d: compose the processing pipeline")

In [3]:
def test_preprocessor():
    prep = Preprocessor()
    text = "To be, or not to be, that is the question"
    assert prep.tokenize(text) == [
        "To", "be", ",", "or", "not", "to", "be", ",", "that", "is", "the", "question"
    ]
    assert prep.stem("retrieval", prep.ps) == "retriev"
    assert prep.is_apt_word("qwerty123") is False
    assert prep.is_apt_word("THE") is False
    assert prep.is_apt_word("café") is True
    assert prep.preprocess(text) == ["or", "not", "question"]
    assert prep.preprocess("SOLAR solar 2024!") == ["solar", "solar"]
    assert prep.preprocess("Space. Energy.") == ["space", "energi"]
    assert prep.preprocess("the AND, 123!") == []
    assert prep.preprocess("") == []

_ = check("E1", test_preprocessor)

E1: INCOMPLETE — E1a: tokenize the text


**Discussion.** What information is lost by removing numbers, punctuation, and stop words? Is stemming the same as correcting spelling?

**Your explanation:**

_Write your answer here._

**Hint.** Consider “2024”, “C++”, “not good”, and a misspelled query.

## 2. Extract article text, not the whole page — E2

The raw response is bytes. Parsed links and extracted article text are separate products; extracting text must never overwrite the raw bytes.

For this collection, use the main `h1` as the title and paragraphs inside an article-body container. Try `[itemprop="articleBody"]` first, then the observed class `.article-body__content`. Do **not** fall back to the first paragraph anywhere on the page: it may be a menu item, promotion, or consent message.

| Selector | Meaning | How to identify it |
|:--|:--|:--|
| `h1` | A level-one heading element | Inspect the article headline in the DOM |
| `[itemprop="articleBody"]` | An element with that exact attribute value | Inspect attributes on the article container |
| `.article-body__content` | An element containing that class token | Inspect a saved page's body wrapper |
| `a[href]` | An anchor that has an `href` attribute | Inspect a link element, not its visual styling |

Selectors come from the document structure; they are not Python keywords. A generated class such as `article-body__content__17Yit` is brittle. Even the cleaner class used here is a snapshot-specific assumption, not a guarantee about the current website. Beautiful Soup's CSS selection and `SoupStrainer` are documented in [4].

### Provided parser shell

`parse("links")` can build a links-only tree with `SoupStrainer`. `parse("text")` builds and caches a full tree but skips link extraction. `parse("both")` shares one full tree. This is **partial parsing for links-only mode**, not a claim that all extraction modes avoid reading the HTML.

In [4]:
EXCLUDED_TAGS = {"script", "style", "noscript", "nav", "footer", "aside", "form"}


def visible_text(tag) -> str:
    """Extract plain text, excluding comments and selected hidden/boilerplate nodes."""
    pieces = []
    for node in tag.find_all(string=True):
        if isinstance(node, Comment) or not str(node).strip():
            continue
        blocked = any(
            parent.name in EXCLUDED_TAGS
            or parent.has_attr("hidden")
            or parent.get("aria-hidden") == "true"
            for parent in node.parents
        )
        if not blocked:
            pieces.append(str(node).strip())
    return " ".join(pieces)


class HtmlArticle:
    def __init__(self, url: str, content: bytes):
        self.url = url
        self.content = content
        self.title = ""
        self.article_text = ""
        self.anchors = []
        self._model = None
        self._text_done = False
        self._links_done = False

    def parse(self, mode="both"):
        if mode not in {"links", "text", "both"}:
            raise ValueError("mode must be links, text, or both")
        if mode in {"text", "both"} and self._model is None:
            self._model = BeautifulSoup(self.content or "", "html.parser")
        if mode in {"links", "both"} and not self._links_done:
            model = self._model
            if model is None:
                model = BeautifulSoup(self.content or "", "html.parser",
                                      parse_only=SoupStrainer("a"))
            self.anchors = [(a.get_text(" ", strip=True), a["href"])
                            for a in model.select("a[href]")]
            self._links_done = True
        if mode in {"text", "both"} and not self._text_done:
            self.title, self.article_text = extract_article_text(self._model)
            self._text_done = True
        return self

### E2 · Implement `extract_article_text`
Return `(title, article_text)`. Use `visible_text` for the title and each paragraph. Combine the title with nonempty body paragraphs, separated by newlines. Multiple body containers may exist; include their paragraphs in document order, but include the same paragraph only once if containers are nested. If the title or usable body paragraphs are missing, return the available title and an empty article text.

**Hint.** Choose one family of body selectors, keep a `seen` set of paragraph identities, and do not mutate the soup. This intentionally extracts the title and paragraphs, not captions, subheadings, or CSS-computed visibility.

In [5]:
def extract_article_text(model) -> tuple[str, str]:
    # TODO E2: find the heading and supported body containers.
    # Collect visible, nonempty paragraphs without double-counting.
    # Return the title and either the joined article text or "".
    raise NotImplementedError("E2: extract the title and body paragraphs")

In [6]:
FIXTURE_HTML = b"""
<html><body>
<nav><p>Menu boilerplate</p></nav>
<h1>Solar energy</h1>
<div class="article-body__content">
  <p>Solar <b>storage</b> improves.</p>
  <p hidden>Hidden message</p>
  <p>Clean energy.<!-- editorial note --><script>tracking()</script></p>
  <div class="article-body__content"><p>Nested paragraph.</p></div>
</div>
<a href="/science/space">Space</a><a>No destination</a>
</body></html>
"""


def test_article_parser():
    doc = HtmlArticle("https://www.nbcnews.com/sample", FIXTURE_HTML)
    doc.parse("links")
    assert doc.anchors == [("Space", "/science/space")]
    assert doc._model is None and doc.article_text == ""
    doc.parse("text")
    assert doc.title == "Solar energy"
    assert doc.article_text == (
        "Solar energy\nSolar storage improves.\nClean energy.\nNested paragraph."
    )
    cached = doc._model
    doc.parse("both")
    assert doc._model is cached and doc.content == FIXTURE_HTML
    no_body = HtmlArticle("https://www.nbcnews.com/category", b"<h1>News</h1><p>Menu</p>")
    assert no_body.parse("text").article_text == ""
    semantic = HtmlArticle("https://www.nbcnews.com/story",
        b'<h1>Wind</h1><div itemprop="articleBody"><p>Wind power.</p></div>')
    assert semantic.parse("text").article_text == "Wind\nWind power."
    assert HtmlArticle("https://www.nbcnews.com/empty", b"").parse("text").article_text == ""

_ = check("E2", test_article_parser)

E2: INCOMPLETE — E2: extract the title and body paragraphs


**Discussion.** Why does a “successful” extraction of every visible string on a page sometimes produce a worse search index?

**Your explanation:**

_Write your answer here._

## 3. URL normalization and crawl scope — E3

For this lab, the allowed origin is `https://www.nbcnews.com` on its default HTTPS port. A prefix check is not sufficient: `https://www.nbcnews.com.evil.test/` begins with the same characters but has a different hostname.

A URL has separate components: scheme, host, path, query, and fragment. Resolve relative links against the current page, verify the parsed host, remove the fragment, and preserve the query string. A fragment points within a page; it should not produce another document ID. Different query strings may genuinely identify different content, so do not drop them indiscriminately. See Python's URL parsing documentation [5].

### E3 · Implement `normalize_url` 

Return an accepted absolute URL, or `None`. Reject missing/blank links, non-HTTPS schemes, user credentials, other hosts, non-default ports, and the listed non-HTML suffixes. Match suffixes against the decoded path, not the whole URL; a PDF with `?download=1` is still a PDF. Normalize an empty path to `/` and remove an explicit port `443`.

**Hint.** Use `urljoin`, `urlsplit`, `urlunsplit`, and `unquote`. Accessing an invalid port may raise `ValueError`. URL suffix filtering is only an early heuristic; a live downloader must also check the HTTP content type.

In [7]:
ALLOWED_HOST = "www.nbcnews.com"
BLOCKED_SUFFIXES = {
    ".pdf", ".mp3", ".avi", ".mp4", ".txt", ".jpg", ".jpeg", ".png",
    ".gif", ".webp", ".svg", ".zip", ".css", ".js", ".xml",
}


def normalize_url(href, base_url: str) -> str | None:

    # TODO E3: resolve, parse, validate, strip the fragment, and rebuild.
    raise NotImplementedError("E3: normalize and filter one URL")

In [8]:
def test_urls():
    base = "https://www.nbcnews.com/news/page"
    assert normalize_url("/science/space#section", base) == "https://www.nbcnews.com/science/space"
    assert normalize_url("next?edition=1#top", base) == "https://www.nbcnews.com/news/next?edition=1"
    assert normalize_url("https://WWW.NBCNEWS.COM:443", base) == "https://www.nbcnews.com/"
    assert normalize_url("#top", base) == base
    for bad in [None, "", "  ", "mailto:editor@example.org", "javascript:void(0)",
                "http://www.nbcnews.com/news", "https://www.nbcnews.com.evil.test/x",
                "https://user@www.nbcnews.com/x", "https://www.nbcnews.com:444/x",
                "https://www.nbcnews.com:bad/x", "/file.PDF?download=1", "/image.%70ng"]:
        assert normalize_url(bad, base) is None, bad

_ = check("E3", test_urls)

E3: INCOMPLETE — E3: normalize and filter one URL


**Discussion.** Four saved files differ only by a URL fragment. Should they receive four independent document IDs? What does URL deduplication fail to detect?

**Your explanation:**

_Write your answer here._

## 4. From documents to postings — E4

Let $D$ be the set of indexed documents, $N=|D|$, and $V$ the vocabulary. Write the processed sequence for document $d$ as $P(d)=(t_1,\ldots,t_{L_d})$.

$$
\operatorname{tf}(t,d)=\sum_{i=1}^{L_d}\mathbf{1}[t_i=t],
\qquad
\operatorname{df}(t)=\sum_{d\in D}\mathbf{1}[\operatorname{tf}(t,d)>0],
\qquad
\operatorname{cf}(t)=\sum_{d\in D}\operatorname{tf}(t,d).
$$

The indicator $\mathbf{1}[A]$ is one when statement $A$ is true and zero otherwise. **Term frequency** counts occurrences in one document; **document frequency** counts documents containing the term; **collection frequency** counts occurrences across the whole collection.

An inverted index maps a term to its postings list: the documents containing that term and its frequency in each document [1]. We retain the starter's compact representation:

```python
index[term] = [collection_frequency, (doc_id, term_frequency), ...]
doc_urls[doc_id] = original_url
doc_lengths[doc_id] = number_of_processed_tokens
```

Thus `index[t][0]` is $\operatorname{cf}(t)$, while `len(index[t]) - 1` is $\operatorname{df}(t)$. The first list element is metadata, **not** a posting. The frequency inside a posting is term frequency, not document frequency.

### A four-document collection

| ID | Text | Processed tokens |
|:--|:--|:--|
| 0 | solar solar energy clean | `solar solar energi clean` |
| 1 | solar energy storage | `solar energi storag` |
| 2 | wind energy clean | `wind energi clean` |
| 3 | wind storage | `wind storag` |

Before running code, build the posting for `solar` and distinguish its three frequency quantities.

**Your posting for `solar`:** `________________________________________`

| Quantity | Your value | What is being counted? |
|:--|:--|:--|
| $\operatorname{tf}(\text{solar},0)$ | ______ | ____________________ |
| $\operatorname{df}(\text{solar})$ | ______ | ____________________ |
| $\operatorname{cf}(\text{solar})$ | ______ | ____________________ |
| Total processed tokens | ______ | ____________________ |
| Total postings | ______ | ____________________ |

### E4 · Implement `InvertedIndex.index_doc` 

Parse and preprocess the document, count its terms, and update the three dictionaries. Each term has **one posting per document**, even when it occurs repeatedly. Add a document's full term count to collection frequency.

Document IDs must be nonnegative integers inserted in strictly increasing order, which keeps each postings list sorted without a later sorting pass. Reject a repeated or out-of-order ID with `ValueError` before changing the index. Skip documents with no processed tokens and return `False`; otherwise return `True`. A skipped document must not consume an ID or add a URL.

**Hint.** `Counter(tokens)` aggregates repeated terms within a document. `len(tokens)` is the document length; `len(Counter(tokens))` is not. The insertion policy is append-only: replacing an indexed document would require removing its old contributions first.

In [9]:
class InvertedIndex:
    def __init__(self):
        self.prep = Preprocessor()
        self.index = {}
        self.doc_urls = {}
        self.doc_lengths = {}
        self._last_doc_id = -1

    def index_doc(self, doc, doc_id: int) -> bool:

        # TODO E4: validate the ID, parse, preprocess, and count terms.
        # Skip empty processed documents without mutating any dictionary.
        # Update the document dictionaries and each term's cf/posting.
        raise NotImplementedError("E4: add one document to the inverted index")

In [10]:
# A small text-only document adapter isolates indexing tests from HTML extraction.
class TextDocument:
    def __init__(self, text, doc_id):
        self.url = f"https://www.nbcnews.com/lab/document-{doc_id}"
        self.article_text = text

    def parse(self, mode="text"):
        return self


TOY_TEXTS = [
    "solar solar energy clean",
    "solar energy storage",
    "wind energy clean",
    "wind storage",
]

# Test fixtures describe the data representation, not its construction algorithm.
TOY_LENGTHS = {0: 4, 1: 3, 2: 3, 3: 2}
TOY_INDEX = {
    "solar": [3, (0, 2), (1, 1)],
    "energi": [3, (0, 1), (1, 1), (2, 1)],
    "clean": [2, (0, 1), (2, 1)],
    "storag": [2, (1, 1), (3, 1)],
    "wind": [2, (2, 1), (3, 1)],
}


def make_toy_index():
    obj = InvertedIndex()
    for doc_id, text in enumerate(TOY_TEXTS):
        assert obj.index_doc(TextDocument(text, doc_id), doc_id)
    return obj


def test_indexing():
    obj = make_toy_index()
    assert obj.index == TOY_INDEX
    assert obj.doc_lengths == TOY_LENGTHS
    validate_index(obj)
    before = copy.deepcopy((obj.index, obj.doc_urls, obj.doc_lengths))
    try:
        obj.index_doc(TextDocument("changed", 0), 0)
    except ValueError:
        pass
    else:
        raise AssertionError("A repeated document ID was accepted.")
    assert (obj.index, obj.doc_urls, obj.doc_lengths) == before
    assert obj.index_doc(TextDocument("the and 123", 4), 4) is False
    assert (obj.index, obj.doc_urls, obj.doc_lengths) == before
    assert obj.index_doc(TextDocument("solar", 4), 4) is True
    assert obj.index["solar"] == [4, (0, 2), (1, 1), (4, 1)]
    validate_index(obj)

_ = check("E4", test_indexing, requires=("E1",))

E4: BLOCKED — first complete E1.


### Invariants: equations that become debugging checks

$$
L_d=\sum_{t\in V}\operatorname{tf}(t,d),\qquad
\operatorname{cf}(t)=\sum_{(d,f)\in\operatorname{postings}(t)}f.
$$

$$
\sum_{d\in D}L_d=\sum_{t\in V}\operatorname{cf}(t),\qquad
\#\text{postings}=\sum_{t\in V}\operatorname{df}(t),\qquad
0<\operatorname{df}(t)\leq\operatorname{cf}(t).
$$

`validate_index` checks these relationships, unique ascending posting IDs, and consistency between the two document dictionaries. Inspect its source in `lab_support.py`; validation is provided infrastructure, not an exercise solution.

For $T$ processed tokens, $P$ postings, and $|V|$ terms, the indexing updates take expected $O(T)$ time with hash-based dictionaries, excluding HTML parsing and the cost of tokenization/stemming. Stored postings require $O(P)$ space, with additional vocabulary, document, and string overhead. A dense document–term matrix would allocate $N|V|$ entries even for absent terms.

**Discussion.** What changes if `index_doc` is called twice for the same document ID without removing its previous contribution?

**Your explanation:**

_Write your answer here._

## 5. Collection statistics and an auditable build — E5

The mean processed document length is

$$
\operatorname{avgdl}=\frac{1}{N}\sum_{d\in D}L_d.
$$

Our policy excludes empty processed documents from $D$. For an empty index, report zero documents, zero vocabulary, zero tokens, zero postings, and an average length of `0.0` rather than dividing by zero.

### E5 · Implement `index_statistics`

Return a dictionary with keys `documents`, `vocabulary`, `total_tokens`, `total_postings`, `avgdl`, `top_df`, and `top_cf`. The last two values are lists of `(term, count)` pairs, limited to `top_n`. Sort by descending count, then ascending term to break ties reproducibly. Reject a negative `top_n`.

**Hint.** The number of postings for a term is one less than the length of its stored entry. Count only indexed documents, not every downloaded or saved file.

In [11]:
def index_statistics(indexer, top_n=10) -> dict:
    # TODO E5: return all seven statistics and deterministic top-term lists.
    raise NotImplementedError("E5: calculate collection statistics")

In [12]:
def test_statistics():
    fixture = SimpleNamespace(index=TOY_INDEX, doc_lengths=TOY_LENGTHS)
    stats = index_statistics(fixture, top_n=2)
    assert stats == {
        "documents": 4, "vocabulary": 5, "total_tokens": 12,
        "total_postings": 11, "avgdl": 3.0,
        "top_df": [("energi", 3), ("clean", 2)],
        "top_cf": [("energi", 3), ("solar", 3)],
    }
    empty = SimpleNamespace(index={}, doc_lengths={})
    assert index_statistics(empty)["avgdl"] == 0.0
    assert index_statistics(empty)["top_df"] == []
    assert index_statistics(fixture, 0)["top_cf"] == []
    try:
        index_statistics(fixture, -1)
    except ValueError:
        pass
    else:
        raise AssertionError("Negative top_n must be rejected.")

_ = check("E5", test_statistics)

E5: INCOMPLETE — E5: calculate collection statistics


### Build the supplied collection

The manifest keeps the original URL separately from a short numeric filename. The original practice of replacing slashes with underscores is not generally reversible and can create very long filenames. Each manifest record also stores the original filename and a SHA-256 checksum; the HTML bytes are unchanged.

The build below processes manifest rows in a fixed order. It keeps the **first** snapshot for each normalized canonical URL, skips empty files and unsupported article layouts, and records a mutually exclusive reason for every row. This is URL-level deduplication, not content deduplication. The `test_collection` is a second set of crawled pages, not a held-out relevance benchmark.

In [13]:
def build_saved_collection(collection="docs_collection"):
    obj = InvertedIndex()
    seen_urls = set()
    audit = Counter({key: 0 for key in (
        "indexed", "empty_file", "duplicate_url", "rejected_url", "no_article", "empty_tokens"
    )})
    for row in manifest_records(collection):
        content = snapshot_bytes(row)
        if not content:
            audit["empty_file"] += 1
            continue
        url = normalize_url(row["url"], row["url"])
        if url is None:
            audit["rejected_url"] += 1
            continue
        if url in seen_urls:
            audit["duplicate_url"] += 1
            continue
        seen_urls.add(url)
        doc = HtmlArticle(url, content).parse("text")
        if not doc.article_text:
            audit["no_article"] += 1
            continue
        if obj.index_doc(doc, len(obj.doc_urls)):
            audit["indexed"] += 1
        else:
            audit["empty_tokens"] += 1
    assert sum(audit.values()) == len(manifest_records(collection))
    validate_index(obj)
    return obj, dict(audit)


def show_table(headers, rows):
    def cell(value):
        return html.escape(str(value)).replace("|", "&#124;").replace("\n", " ")
    lines = ["| " + " | ".join(map(cell, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(cell, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


corpus_index = None
if ready("E1", "E2", "E3", "E4", "E5"):
    corpus_index, corpus_audit = build_saved_collection()
    show_table(["Build outcome", "Snapshots"], corpus_audit.items())
    corpus_stats = index_statistics(corpus_index)
    show_table(["Statistic", "Value"],
               [(key, round(value, 3) if isinstance(value, float) else value)
                for key, value in corpus_stats.items() if not key.startswith("top_")])
    show_table(["Term", "Document frequency"], corpus_stats["top_df"])

Not run: first complete E1, E2, E3, E4, E5.


**Discussion.** Why is the number of indexed documents smaller than the number of files? How might retaining duplicate snapshots change BM25?

**Your explanation:**

_Write your answer here._

**Hint.** Use the actual audit counts, rather than assuming every saved page is an article.

## 6. Boolean AND retrieval — E6

Let $Q$ be the **set of distinct normalized query terms** and let

$$
D_t=\{d\in D:\operatorname{tf}(t,d)>0\}.
$$

AND retrieval returns documents that contain every query term:

$$
R_{\mathrm{AND}}(Q)=\bigcap_{t\in Q}D_t.
$$

An unknown term has an empty posting set, so a query containing one must return no AND results. Do not silently discard unknown terms. For an empty processed query, this lab's user-facing policy also returns an empty set; this is a deliberate convention, not the mathematical empty-intersection identity.

For the toy collection, `solar energy` means `solar AND energi`. The posting sets are $\{0,1\}$ and $\{0,1,2\}$, so the result is $\{0,1\}$. The result is a **set**, not a relevance ranking.

### E6 · Implement `boolean_retrieval`

The prepared query is a `Counter` or dictionary `{term: frequency}`. Ignore entries with nonpositive counts. Repeated query terms do not change an AND result. Process terms in increasing document frequency and stop if the intersection becomes empty. The dictionary must not be modified.

**Hint.** Build sets from `entry[1:]`, not the entire entry. Check for an absent term before retrieving its postings. Intersecting shorter lists first can reduce intermediate work [1].

In [14]:
def prepare_query(raw_query: str) -> Counter:
    return Counter(Preprocessor().preprocess(raw_query))

In [15]:
def boolean_retrieval(query: dict, index: dict) -> set[int]:
    # TODO E6: handle empty/unknown terms, then intersect posting sets.
    raise NotImplementedError("E6: retrieve documents containing every query term")

In [16]:
def test_boolean_retrieval():
    before = copy.deepcopy(TOY_INDEX)
    assert boolean_retrieval({"solar": 1, "energi": 1}, TOY_INDEX) == {0, 1}
    assert boolean_retrieval({"solar": 5}, TOY_INDEX) == {0, 1}
    assert boolean_retrieval({"solar": 1, "unknown": 1}, TOY_INDEX) == set()
    assert boolean_retrieval({"unknown": 1}, TOY_INDEX) == set()
    assert boolean_retrieval({}, TOY_INDEX) == set()
    assert boolean_retrieval({"solar": 0}, TOY_INDEX) == set()
    assert boolean_retrieval({"solar": 1}, {}) == set()
    assert boolean_retrieval({"clean": 1, "storag": 1}, TOY_INDEX) == set()
    assert TOY_INDEX == before

_ = check("E6", test_boolean_retrieval)

E6: INCOMPLETE — E6: retrieve documents containing every query term


**Discussion.** For the query `solar unknownterm`, why is it correct for AND retrieval to return nothing even though two documents contain “solar”? What would OR retrieval return?

**Your explanation:**

_Write your answer here._

## 7. Ranked retrieval with BM25 — E7

AND retrieval treats all matching documents equally. BM25 adds term-frequency saturation, a rarity weight, and document-length normalization. This lab uses the following explicit variant [2]:

$$
\operatorname{BM25}(d,Q)=
\sum_{t\in Q}
\operatorname{IDF}(t)\,
\frac{\operatorname{tf}(t,d)(k_1+1)}
{\operatorname{tf}(t,d)+k_1\left(1-b+b\frac{L_d}{\operatorname{avgdl}}\right)}.
$$

$$
\operatorname{IDF}(t)=\ln\left(1+\frac{N-\operatorname{df}(t)+0.5}
{\operatorname{df}(t)+0.5}\right).
$$

We use the positive, smoothed IDF form documented by Apache Lucene [2], with the natural logarithm. The complete scoring function shown here is a pedagogical BM25 formula; it is not a claim of bit-for-bit equivalence to Lucene's implementation. $Q$ contains each distinct normalized query term once: repeated query terms do **not** multiply the score in this lab.

| Symbol | Meaning in code |
|:--|:--|
| $N$ | `len(doc_lengths)`; indexed, nonempty documents |
| $\operatorname{df}(t)$ | `len(index[t]) - 1`; not collection frequency |
| $\operatorname{tf}(t,d)$ | The second value in posting `(doc_id, tf)` |
| $L_d$ | `doc_lengths[doc_id]`; processed token count |
| $\operatorname{avgdl}$ | Mean processed length across the indexed collection |
| $k_1$ | Nonnegative saturation parameter; default `1.2` |
| $b$ | Length-normalization strength in $[0,1]$; default `0.75` |

### Reading the equation

For fixed document length, the term-frequency fraction increases with frequency but approaches $k_1+1$ rather than growing without bound. With $k_1=0$, any present term contributes its IDF once. With $b=0$, length normalization disappears. With $b=1$, the normalization factor uses the full length ratio. At $L_d=\operatorname{avgdl}$, the length term is simply $k_1$.

BM25 considers the **union** of postings for known query terms, not just the AND result. An unknown term contributes nothing. A document matching one query term can therefore receive a score, even when AND retrieval would reject it. Scores are ranking values, not probabilities, and should not be compared as calibrated confidence across unrelated queries.

### Hand calculation: `solar energy`

For the toy collection, $N=4$, $\operatorname{avgdl}=3$, $\operatorname{df}(\text{solar})=2$, and $\operatorname{df}(\text{energi})=3$. Use $k_1=1.2$ and $b=0.75$.

$$
\operatorname{IDF}(\text{solar})=\ln(2),\qquad
\operatorname{IDF}(\text{energi})=\ln(10/7).
$$

For document 0, first compute the length term

$$
K_0=1.2\left(1-0.75+0.75\frac{4}{3}\right).
$$

Then substitute $\operatorname{tf}(\text{solar},0)=2$ and $\operatorname{tf}(\text{energi},0)=1$ separately. Round only the final displayed values.

| Document | Length term $K_d$ | Solar contribution | Energy contribution | Total score |
|:--|:--|:--|:--|:--|
| 0 | ______ | ______ | ______ | ______ |
| 1 | ______ | ______ | ______ | ______ |
| 2 | ______ | ______ | ______ | ______ |
| 3 | ______ | ______ | ______ | ______ |

**Predicted ranking of documents with at least one match:** ____________________

**Why can the ranked result contain a document absent from the AND result?**

_Write your answer here._

### E7 · Implement `okapi_scoring`

Return `{doc_id: score}` for documents with at least one known query term. Use each positive-count query term once. Return `{}` for an empty query, an empty collection, a collection with zero average length, or a query with no known terms. Use `math.log1p(ratio)` for the stated IDF. Reject nonfinite/negative `k1` and nonfinite `b` outside `[0, 1]`.

The index is assumed to have passed `validate_index`; do not scan every document for every term. Accumulate contributions by traversing the relevant postings. Do not round stored scores, and do not treat dictionary insertion order as rank order.

**Hint.** Compute `N` and `avgdl` once, `df` and IDF once per query term, and the length-dependent fraction once per posting. Use `scores.get(doc_id, 0.0)` to accumulate a new contribution.

In [17]:
def okapi_scoring(query: dict, doc_lengths: dict, index: dict,
                  k1=1.2, b=0.75) -> dict[int, float]:
    # TODO E7: validate parameters and handle the empty-collection cases.
    # For each known, positive-count query term, compute its IDF.
    # Traverse its postings and accumulate BM25 contributions.
    raise NotImplementedError("E7: calculate BM25 scores")

In [18]:
def rank_scores(scores: dict, top_k=5):
    """Rank descending by score; break exact ties by ascending document ID."""
    if type(top_k) is not int or top_k < 0:
        raise ValueError("top_k must be a nonnegative integer")
    return sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))[:top_k]


def test_bm25():
    q = {"solar": 1, "energi": 1}
    scores = okapi_scoring(q, TOY_LENGTHS, TOY_INDEX)
    expected = {0: 1.1852589776557300, 1: 1.0498221244986776,
                2: 0.3566749439387324}
    assert set(scores) == set(expected)
    for doc_id, value in expected.items():
        assert math.isclose(scores[doc_id], value, rel_tol=1e-12)
    assert [doc_id for doc_id, score in rank_scores(scores)] == [0, 1, 2]
    assert okapi_scoring({"solar": 8, "energi": 1}, TOY_LENGTHS, TOY_INDEX) == scores
    assert okapi_scoring({**q, "unknown": 1}, TOY_LENGTHS, TOY_INDEX) == scores
    assert okapi_scoring({}, TOY_LENGTHS, TOY_INDEX) == {}
    assert okapi_scoring({"unknown": 1}, TOY_LENGTHS, TOY_INDEX) == {}
    assert okapi_scoring({"solar": 0}, TOY_LENGTHS, TOY_INDEX) == {}
    assert okapi_scoring(q, {}, {}) == {}
    assert okapi_scoring(q, {0: 0}, {}) == {}
    presence = okapi_scoring({"solar": 1}, TOY_LENGTHS, TOY_INDEX, k1=0)
    assert math.isclose(presence[0], presence[1], rel_tol=1e-12)
    equal_tf = okapi_scoring({"energi": 1}, TOY_LENGTHS, TOY_INDEX, b=0)
    assert len(set(equal_tf.values())) == 1
    # cf can exceed N; IDF must still depend on df, not cf.
    common = okapi_scoring({"x": 1}, {0: 8, 1: 2}, {"x": [10, (0, 8), (1, 2)]})
    assert all(math.isfinite(v) and v > 0 for v in common.values())
    assert rank_scores({4: 1.0, 2: 1.0}) == [(2, 1.0), (4, 1.0)]
    for kwargs in [{"k1": -1}, {"k1": float("nan")}, {"b": 1.1}, {"b": float("inf")}]:
        try:
            okapi_scoring(q, TOY_LENGTHS, TOY_INDEX, **kwargs)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid parameters accepted: {kwargs}")

_ = check("E7", test_bm25)

E7: INCOMPLETE — E7: calculate BM25 scores


### Parameter experiment

Compare the toy query across the settings below. Observe both the scores and the order. A parameter change need not change the ranking of this particular collection.

In [19]:
if ready("E7"):
    rows = []
    for k1, b in [(0.0, 0.75), (1.2, 0.0), (1.2, 0.75), (1.2, 1.0), (2.0, 0.75)]:
        scores = okapi_scoring({"solar": 1, "energi": 1}, TOY_LENGTHS, TOY_INDEX, k1=k1, b=b)
        rows.append([k1, b, ", ".join(str(d) for d, _ in rank_scores(scores)),
                     f"{scores[0]:.6f}", f"{scores[1]:.6f}"])
    show_table(["k1", "b", "Ranking", "Score d0", "Score d1"], rows)

Not run: first complete E7.


**Discussion.** When `k1=0`, why do documents 0 and 1 tie? When `b=0`, does the ranking ignore term frequency as well as document length?

**Your explanation:**

_Write your answer here._

## 8. Breadth-first crawling connected to the index — E8

The crawler takes a `fetch(url)` function that returns HTML bytes or `None`. Supplying the function as an argument separates **which page to visit next** from **how bytes are obtained**. The tests use an in-memory miniature web, so they are deterministic and do not contact a news site.

### Crawl contract

The source is at depth 0. `max_depth=0` visits only the source; `max_depth=1` also visits its direct links. Never visit beyond the maximum depth. Use a FIFO queue for breadth-first order. Mark a normalized URL as discovered when it enters the queue, so cycles and repeated links cannot create duplicate work.

`limit` counts **successfully indexed articles**, not fetched pages. `limit=0` performs no fetch. `max_visits` separately limits attempted fetches, because a site may have many non-article pages. Print each visited URL. A fetch returning `None` is a recoverable failure; do not hide programming errors with a bare `except`.

### E8 · Implement `crawl_generator_for_index`

Yield each article immediately after it is indexed. Fetch and inspect non-article pages too: their links may lead to articles. Use `normalize_url`, `HtmlArticle`, and `indexer.index_doc`. When `collection_path` is supplied, use `persist_page` to save the raw bytes of indexed articles with a fixed-length filename and a separate URL record. Saving the final index is explicit in the next section, not a side effect that may be skipped when a generator is only partially consumed.

**Hint.** Store `(url, depth)` pairs in a `deque`. The next document ID is `indexer._last_doc_id + 1`. `yield` pauses execution; any work placed after it resumes only when the caller asks for another item.

In [20]:
def crawl_generator_for_index(indexer, source, max_depth, fetch, *,
                              limit=None, collection_path=None, max_visits=100):

    # TODO E8: validate bounds and handle limit=0 before any fetch.
    # Traverse normalized URLs in BFS order with a discovered set.
    # Index/yield articles, preserve raw pages if requested, and enqueue links.
    raise NotImplementedError("E8: connect breadth-first crawling to indexing")
    yield  # Keeps this scaffold a generator; replace it with the real loop.

In [21]:
def miniature_web():
    origin = "https://www.nbcnews.com"
    def article(title, text, links=""):
        return (f'<h1>{title}</h1><div class="article-body__content"><p>{text}</p></div>'
                + links).encode("utf-8")
    return {
        origin + "/lab": (
            b'<h1>Lab links</h1><a href="/lab/a">A</a><a href="/lab/b">B</a>'
            b'<a href="/lab/a#again">A again</a><a href="/file.pdf?q=1">PDF</a>'
            b'<a href="https://www.nbcnews.com.evil.test/x">Outside</a>'
        ),
        origin + "/lab/a": article("Solar", "Clean solar energy.",
            '<a href="/lab/c">C</a><a href="/lab">Back</a>'),
        origin + "/lab/b": article("Wind", "Wind storage.", '<a href="/lab/c#same">C again</a>'),
        origin + "/lab/c": article("Storage", "Energy storage."),
    }


def test_crawler():
    web = miniature_web()
    source = "https://www.nbcnews.com/lab"
    def run(depth, limit=None, max_visits=100, collection_path=None):
        visited = []
        def fetch(url):
            visited.append(url)
            return web.get(url)
        obj = InvertedIndex()
        docs = list(crawl_generator_for_index(
            obj, source, depth, fetch, limit=limit,
            max_visits=max_visits, collection_path=collection_path))
        validate_index(obj)
        return obj, docs, visited
    assert run(0)[2] == [source]
    assert len(run(0)[1]) == 0
    assert [doc.url for doc in run(1)[1]] == [source + "/a", source + "/b"]
    obj, docs, visited = run(2)
    assert visited == [source, source + "/a", source + "/b", source + "/c"]
    assert len(docs) == 3 and len(obj.doc_urls) == 3
    assert len(run(2, limit=1)[1]) == 1
    assert run(2, limit=0)[2] == []
    assert len(run(2, max_visits=1)[2]) == 1
    with TemporaryDirectory() as folder:
        saved, docs, _ = run(1, limit=1, collection_path=folder)
        files = list(Path(folder).glob("*.html"))
        assert len(files) == 1 and files[0].read_bytes() == docs[0].content
        assert len(list(Path(folder).glob("*.json"))) == 1
    broken_web = dict(web)
    broken_web[source + "/a"] = None
    obj = InvertedIndex()
    docs = list(crawl_generator_for_index(obj, source, 2, broken_web.get))
    assert [doc.url for doc in docs] == [source + "/b", source + "/c"]

# Capture repeated test traces; a single readable trace is shown below.
from contextlib import redirect_stdout
import io

def quiet_crawler_test():
    with redirect_stdout(io.StringIO()):
        test_crawler()

_ = check("E8", quiet_crawler_test, requires=("E1", "E2", "E3", "E4"))

if ready("E8"):
    demo_index = InvertedIndex()
    pages = miniature_web()
    for doc in crawl_generator_for_index(
            demo_index, "https://www.nbcnews.com/lab", 2, pages.get, limit=2):
        print("Indexed:", doc.title)

E8: BLOCKED — first complete E1, E2, E3, E4.
Not run: first complete E8.


**Discussion.** Why is `limit=2` not equivalent to `max_visits=2`? Why should duplicate URLs enter the discovered set when enqueued rather than only when fetched?

**Your explanation:**

_Write your answer here._

### Optional live adapter — not part of the required run

The required lab uses saved or in-memory pages. A live adapter is supplied in `live_fetch.py` for an instructor-supervised extension. It checks the allowed origin and `robots.txt`, uses timeouts and a delay, rejects redirects rather than following them outside the policy, accepts HTML content types only, caps response size, and reports request failures. `robots.txt` is a crawler instruction mechanism, not proof of permission; confirm the site's current terms and obtain any required authorization before running a live crawl [6, 7].

The cell below is commented out intentionally. Do not enable it merely to make a test pass. An unavailable or changed live site does not invalidate the offline exercises.

In [22]:
# Optional extension only; not executed during Run All.
# from live_fetch import PoliteHtmlFetcher
# live_index = InvertedIndex()
# with PoliteHtmlFetcher(
#         user_agent="CourseLab04/1.0 (your real course contact)", delay=2.0) as fetch:
#     for article in crawl_generator_for_index(
#             live_index, "https://www.nbcnews.com/news", 1, fetch,
#             limit=3, max_visits=10, collection_path=ROOT / "outputs" / "live_pages"):
#         print("Indexed:", article.url)
# save_index(live_index, ROOT / "outputs" / "live_index")

## 9. Save, reload, and search

Persist all three dictionaries together with the preprocessing configuration. The supplied `save_index` and `load_index` functions use readable JSON and checksum metadata. JSON turns integer dictionary keys into strings and tuples into arrays, so loading must restore integer document IDs and tuple postings. The loader rejects a mismatched configuration or checksum, then validates the index [8].

Checksums detect accidental changes, not a maliciously replaced index and manifest. This teaching format is not a transactional database and is not intended for concurrent writers. Unlike `pickle`, loading JSON does not execute arbitrary Python objects.

The following check covers both collections and a round trip of the main index. Corpus document counts are reported after the declared extraction and deduplication rules are applied.

In [23]:
def test_integration():
    global corpus_index
    if corpus_index is None:
        corpus_index, _ = build_saved_collection()
    saved_folder = ROOT / "outputs" / "saved_index"
    save_index(corpus_index, saved_folder)
    restored = load_index(InvertedIndex, saved_folder)
    assert restored.index == corpus_index.index
    assert restored.doc_urls == corpus_index.doc_urls
    assert restored.doc_lengths == corpus_index.doc_lengths
    assert all(type(doc_id) is int for doc_id in restored.doc_urls)
    q = prepare_query("space mission")
    assert boolean_retrieval(q, restored.index) == boolean_retrieval(q, corpus_index.index)
    assert okapi_scoring(q, restored.doc_lengths, restored.index) == okapi_scoring(
        q, corpus_index.doc_lengths, corpus_index.index)
    test_index, test_audit = build_saved_collection("test_collection")
    assert sum(test_audit.values()) == 28
    assert test_audit["empty_file"] == 1
    assert test_audit["duplicate_url"] == 4
    assert test_index.doc_urls
    print("Main collection indexed documents:", len(corpus_index.doc_urls))
    print("Test collection indexed documents:", len(test_index.doc_urls))
    print("Saved and reloaded: outputs/saved_index")

_ = check("Integration", test_integration, requires=("E1", "E2", "E3", "E4", "E5", "E6", "E7"))

Integration: BLOCKED — first complete E1, E2, E3, E4, E5, E6, E7.


### Search the saved news collection

Normalize each query once, show the AND result count, then show the five highest BM25 scores. Sorting is explicit and ties use document ID. The titles below are read from the saved pages; no live page requests are made.

Try a correctly spelled query, a repeated-term query, and an unknown-term query. Changing `top_k` changes how many ranked matches are displayed, not the underlying scores. The function does not interpret quotation marks as a phrase query.

In [24]:
def search_saved_collection(raw_query, top_k=5):
    if corpus_index is None:
        raise RuntimeError("Build the saved collection first.")
    query = prepare_query(raw_query)
    and_matches = boolean_retrieval(query, corpus_index.index)
    scores = okapi_scoring(query, corpus_index.doc_lengths, corpus_index.index)
    print(f"Query: {raw_query!r}; processed: {dict(query)}")
    print(f"AND matches: {len(and_matches)}; BM25 candidates: {len(scores)}")
    by_url = {}
    for row in manifest_records():
        by_url.setdefault(row["url"], row)
    rows = []
    for rank, (doc_id, score) in enumerate(rank_scores(scores, top_k), start=1):
        url = corpus_index.doc_urls[doc_id]
        article = HtmlArticle(url, snapshot_bytes(by_url[url])).parse("text")
        rows.append([rank, doc_id, f"{score:.6f}", article.title])
    if rows:
        show_table(["Rank", "Document ID", "BM25", "Saved article title"], rows)
    else:
        print("No ranked matches.")
    return scores


if ready("Integration"):
    for query_text in ["space mission", "skin care", "solar unknownterm"]:
        _ = search_saved_collection(query_text)

Not run: first complete Integration.


**Discussion.** Choose two queries and compare their AND and ranked results. Explain one unexpected match or missing match using the pipeline, not just a judgment that the engine is “good” or “bad”.

**Your explanation:**

_Write your answer here._

**Hint.** Report the exact raw query, normalized terms, and a document ID or title.

## 10. Completion checklist and exit questions

A complete submission has **E1–E8 and Integration marked PASS**, includes the hand calculations and written explanations, and runs after a fresh kernel restart. Keep the exercise tests unchanged and retain the saved index under `outputs/saved_index`. Submit the completed notebook and saved-index folder; the large unchanged snapshot collection need not be resubmitted unless requested by your instructor.

Answer these exit questions in a few sentences each. The hand calculation, search explanation, and written responses together are worth 5 marks:

1. Give a concrete example where tf, df, and cf differ, and explain why df rather than cf belongs in the stated IDF.
2. Identify one case where Boolean AND and BM25 intentionally return different document sets.
3. Explain why changing the preprocessing policy requires rebuilding an existing index.
4. What extra index data would be needed for exact phrase search? Why can this index not recover it?
5. Name one limitation of URL deduplication and one limitation of judging search quality from the top-frequency terms.

**Responses**

1. _Write your answer here._
2. _Write your answer here._
3. _Write your answer here._
4. _Write your answer here._
5. _Write your answer here._

In [25]:
show_table(["Check", "Status"], [(name, CHECKS.get(name, "NOT RUN"))
    for name in ["E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "Integration"]])
if all(CHECKS.get(name) == "PASS" for name in [
        "E1", "E2", "E3", "E4", "E5", "E6", "E7", "E8", "Integration"]):
    print("All required code checks passed. Review the written responses before submission.")
else:
    print("The workbook is not complete. Resolve incomplete or blocked exercises, then restart and run all.")

| Check | Status |
| --- | --- |
| E1 | INCOMPLETE |
| E2 | INCOMPLETE |
| E3 | INCOMPLETE |
| E4 | BLOCKED |
| E5 | INCOMPLETE |
| E6 | INCOMPLETE |
| E7 | INCOMPLETE |
| E8 | BLOCKED |
| Integration | BLOCKED |

The workbook is not complete. Resolve incomplete or blocked exercises, then restart and run all.


## Optional extensions

**Sorted-list intersection.** Replace set-based AND retrieval with a two-pointer merge. Compare both implementations on identical queries and confirm identical result sets before measuring speed.

**Positional indexing.** Extend a posting to store positions as well as a frequency. Decide whether positions are assigned before or after stop-word removal, and document the consequence for phrase search.

**Gap encoding.** For increasing document IDs $d_1,d_2,\ldots$, define $g_1=d_1$ and $g_i=d_i-d_{i-1}$ for $i>1$. Explain why smaller gaps may be cheaper to encode, while noting that storing Python integers in a list does not itself implement compression.

**Retrieval evaluation.** Write a small set of queries and manually judge relevant documents. Compare two settings on those same judgments. Do not choose a model only because it produces larger raw scores.

## References

[1] Manning, Raghavan, and Schütze, *Introduction to Information Retrieval*, Chapter 1: [index construction](https://nlp.stanford.edu/IR-book/html/htmledition/a-first-take-at-building-an-inverted-index-1.html) and [Boolean query processing](https://nlp.stanford.edu/IR-book/html/htmledition/processing-boolean-queries-1.html).

[2] Manning, Raghavan, and Schütze, [Okapi BM25](https://nlp.stanford.edu/IR-book/html/htmledition/okapi-bm25-a-non-binary-model-1.html); Apache Lucene, [BM25Similarity 9.12.1](https://lucene.apache.org/core/9_12_1/core/org/apache/lucene/search/similarities/BM25Similarity.html), for the positive smoothed IDF definition and parameter interpretation.

[3] NLTK, [regular-expression tokenizers and WordPunctTokenizer](https://www.nltk.org/api/nltk.tokenize.regexp.html).

[4] Beautiful Soup, [documentation: CSS selectors and partial parsing](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

[5] Python documentation, [urllib.parse](https://docs.python.org/3/library/urllib.parse.html).

[6] Requests, [Quickstart: timeouts, response headers, and redirects](https://requests.readthedocs.io/en/latest/user/quickstart/).

[7] Python documentation, [urllib.robotparser](https://docs.python.org/3/library/urllib.robotparser.html).

[8] Python documentation, [JSON encoding and decoding](https://docs.python.org/3/library/json.html).

**Source collection.** The two supplied Lab 04 notebooks and all HTML snapshots from `lab04.zip`. Snapshot URLs, original filenames, and checksums are retained in `data/manifest.json`. Inclusion in this course package does not change the original publishers' ownership or grant broader redistribution rights.